# PLS characteristic importance

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**Description**

1. Change the results path for the number hidden layer in each model, iteratively : results_CA_0, results_CA_1, results_CA_2, results_CA_0
2. Run the code that defines the desired model with the number of hidden layers
3. Run the codes that preprocess the data for the models.
4. Run the code for the cross-validation, which makes prediction on the validation set. The performance evaluation metrcs are saved in the results path.Remember to change the results path for the cross-validation.
5. Run the the code that predicts the desired model with the desired number of hidden layers. remember to change the results path for the out-of-sample  predictions on the test set. This code makes predictions for each latent latent factor across a time-series split, and the predicted results are saved.

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            train_end_idx = test_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits


In [ ]:
!pip install tables

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV


from numpy.linalg import inv
from google.colab import drive




**Setting up TPU**

In [ ]:
# # Set up the TPU
# try:
#     tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
#     print('Running on TPU ', tpu.master())

#     # Always connect to the TPU cluster and initialize the TPU system
#     tf.config.experimental_connect_to_cluster(tpu)
#     tf.tpu.experimental.initialize_tpu_system(tpu)

#     strategy = tf.distribute.TPUStrategy(tpu)
#     print("All TPU devices: ", tf.config.experimental.list_logical_devices('TPU'))
# except ValueError:
#     raise BaseException('ERROR: Not connected to a TPU runtime; select TPU from the "Runtime > Change runtime type" menu.')



**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/Characteristics_importance_PLS')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


In [ ]:

###########################
## MOUNT TO GOOGLE DRIVE ##
###########################

# Mount Google Drive
drive.mount('/content/drive')

################################
## LOAD DATA FOR BETA NETWORK ##
################################


file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'

# Make sure merged_df1 is a DataFrame and not a function or any other type.
merged_df_padded = pd.read_parquet(file_path)

# Assuming merged_df_padded is already defined
# Ensure the date level of the MultiIndex is of datetime type
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)

# Filter the DataFrame to include only rows from 1979 and onwards
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Swap the levels of the MultiIndex
merged_df_padded = merged_df_padded.swaplevel('date', 'permno')

# Optionally, sort the index to ensure correct ordering
merged_df_padded = merged_df_padded.sort_index()

# Remove rows where any column has a value of -2
merged_df_padded = merged_df_padded[~(merged_df_padded == -2).any(axis=1)]

# Now check the new structure
print(merged_df_padded.index)
print(merged_df_padded.shape)
print(type(merged_df_padded))

print(merged_df_padded)
print(merged_df_padded.index)
print(merged_df_padded.shape)
print(type(merged_df_padded))




**MODEL PARAMETER SPECIFICATIONS**

In [ ]:

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
columns_to_select = [f'characteristic_{code}' for code in all_codes]

# Ensure "ret_excess" is included in the columns_to_select
columns_to_select_with_ret_excess = columns_to_select + ["ret_excess"]

# Select the subset of columns, including "ret_excess"
merged_df_padded_subset = merged_df_padded[columns_to_select_with_ret_excess]




n_factors = 6
n_narratives = len(columns_to_select)
n_tickers = len(merged_df_padded.index.unique('permno'))

print(n_narratives)
print(n_tickers)



**Converting data to float16 for efficiency and saving files**

# **Training loop with figure creation. Figures are saved.**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from pathlib import Path
from utils import format_time, MultipleTimeSeriesCV  # Assuming you have this utility function

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(merged_df_padded_subset))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]

    y_train = train_set['ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])

    # Extract date and permno from the multi-index and reset index
    test_dates = test_set.index.get_level_values('date')
    test_permnos = test_set.index.get_level_values('permno')

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    # Also filter the dates and permnos accordingly
    test_dates_filtered = test_dates[~test_all_zeros]
    test_permnos_filtered = test_permnos[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered, test_dates_filtered, test_permnos_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/total_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors to consider for PLS
factors = [5]

# Function to compute permutation feature importance for PLS
def compute_permutation_variable_importance_pls(model, X, y, scaler):
    X_scaled = scaler.transform(X)

    # Get the baseline performance
    preds = model.predict(X_scaled)
    baseline_r2 = r2_score(y, preds)

    importances = []

    # Iterate over each feature
    for i in range(X.shape[1]):
        X_copy = X.copy()

        # Shuffle the i-th feature
        X_copy.iloc[:, i] = np.random.permutation(X_copy.iloc[:, i].values)

        # Scale the permuted data
        X_copy_scaled = scaler.transform(X_copy)

        # Predict with the permuted data
        preds_permuted = model.predict(X_copy_scaled)
        permuted_r2 = r2_score(y, preds_permuted)

        importance = baseline_r2 - permuted_r2
        importances.append(importance)

    return importances

# Initialize lists to store importances
fold_importances_list = []

# Loop over each factor and calculate average R^2 for all folds for PLS and feature importances
for n_factors in tqdm(factors):
    r2_scores_pls = []
    all_predictions_pls = []  # Initialize all_predictions_pls here
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test, test_dates, test_permnos = get_train_test_data(merged_df_padded_subset, train_idx, test_idx)

        # Standardize the predictors
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)

        # Apply PLS
        pls = PLSRegression(n_components=n_factors)
        pls.fit(x_train_scaled, y_train)

        # Predict and evaluate for PLS
        y_pred_pls = pls.predict(x_test_scaled)
        r2_pls = r2_score(y_test, y_pred_pls)

        # Create a DataFrame for the current fold's predictions
        fold_predictions = pd.DataFrame({
            'fold': fold,
            'date': test_dates,
            'permno': test_permnos,
            'actual': y_test,
            'predicted': y_pred_pls.flatten()
        })

        print(f'Average R^2 for PLS with {n_factors}| fold : {fold}| factors: {r2_pls}')

        # Add the fold predictions to the list
        all_predictions_pls.append(fold_predictions)

        # Save R² scores
        r2_scores_pls.append(r2_pls)

        # Calculate feature importance using permutation
        importances = compute_permutation_variable_importance_pls(pls, x_test, y_test, scaler)
        fold_importances_list.append(importances)

    # Combine all fold predictions into a single DataFrame for this factor
    combined_predictions = pd.concat(all_predictions_pls, ignore_index=True)

    # Save combined predictions to HDF5 file
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.put(f'/pls_predictions/factors_{n_factors}', combined_predictions)

    avg_r2_pls = np.mean(r2_scores_pls)
    print(f'Average R^2 for PLS with {n_factors} factors: {avg_r2_pls}')

# Aggregate importances across all folds
avg_importances = np.mean(fold_importances_list, axis=0)
importances_df = pd.DataFrame(avg_importances, index=x_train.columns, columns=['importance'])
importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})

# Save the importance DataFrame
importances_df.to_hdf(str(results_path / 'variable_importance_pls.h5'), key='all_folds', append=True, format='table')

# Sort and select the top 20 variable importances
importances_df = importances_df.sort_values('importance_normalized', ascending=False).head(20)

# Reverse the DataFrame so that the highest importance is at the top in the plot
importances_df = importances_df.iloc[::-1]

# Plot the top 20 variable importances
importances_df.plot(kind='barh', x='variable', y='importance_normalized', legend=False)
plt.xlabel('Normalized Importance')
plt.ylabel('Variables')
plt.title(f'Top 20 Normalized Characteristics Importances - PLS Model')
plt.savefig(str(results_path / f'top_20_importances_pls.png'))
plt.show()


#**Load characteristics importance and create correct plot**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/total_results_PLS')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Ensure the results directory exists
results_path.mkdir(parents=True, exist_ok=True)

# Load the importance DataFrame from the saved HDF5 file generated in Code 1
importances_df = pd.read_hdf(str(base_path / 'variable_importance_pls.h5'), key='all_folds')

# Group by the 'variable' column and average the 'importance' and 'importance_normalized' columns
importances_avg_df = importances_df.groupby('variable', as_index=False).mean()

# Remove the prefix "characteristic_" from the 'variable' column
importances_avg_df['variable'] = importances_avg_df['variable'].str.replace('characteristic_', '', regex=False)

# Sort and select the top 20 variable importances by 'importance_normalized'
top_20_importances_df = importances_avg_df.sort_values('importance_normalized', ascending=False).head(20)

# Reverse the DataFrame so that the highest importance is at the top in the plot
top_20_importances_df = top_20_importances_df.iloc[::-1]

# Plot the top 20 variable importances
top_20_importances_df.plot(kind='barh', x='variable', y='importance_normalized', legend=False)
plt.xlabel('Normalized Importance')
plt.ylabel('Characteristics')

# Save the plot
plt.savefig(str(results_path / 'top_20_importances_pls.png'))
plt.show()


In [ ]:
print(importances_df)

In [ ]:
from google.colab import files

# Path to the saved figure
figure_path = str(results_path / 'top_20_importances_pca.png')

# Download the figure
files.download(figure_path)
